# Spam Classification

In [2]:
import requests
import tarfile

import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import make_pipeline


from bs4 import BeautifulSoup
from pathlib import Path 



- Download and unzip dataset files from `https://spamassassin.apache.org/old/publiccorpus/`

In [3]:
BASE_URL = "https://spamassassin.apache.org/old/publiccorpus/"
DOWNLOAD_DIR = Path("../datasets/spamassassin_corpus")
DOWNLOAD_DIR.mkdir(exist_ok=True)


In [4]:
# Get directory listing
response = requests.get(BASE_URL)
response.raise_for_status()
soup = BeautifulSoup(response.content, "html.parser")

# Download every .tar.bz2 file
for link in soup.find_all("a"):
    href = link.get("href")
    if href and href.endswith(".tar.bz2"):
        url = BASE_URL + href
        out_file = DOWNLOAD_DIR / href

        print(f"Downloading {href}...")

        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(out_file, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

print("Done.")

Done.


In [6]:
for archive in DOWNLOAD_DIR.glob("*.tar.bz2"):
    extract_dir = DOWNLOAD_DIR / archive.stem.replace(".tar", "")
    extract_dir.mkdir(exist_ok=True)

    print(f"Extracting {archive.name}...")
    with tarfile.open(archive, "r:bz2") as tar:
        tar.extractall(extract_dir)

print("Extraction complete.")

Extracting 20030228_hard_ham.tar.bz2...
Extracting 20021010_hard_ham.tar.bz2...
Extracting 20030228_spam_2.tar.bz2...
Extracting 20030228_easy_ham_2.tar.bz2...
Extracting 20030228_spam.tar.bz2...
Extracting 20030228_easy_ham.tar.bz2...
Extracting 20021010_spam.tar.bz2...
Extracting 20021010_easy_ham.tar.bz2...
Extracting 20050311_spam_2.tar.bz2...
Extraction complete.


In [4]:
rows = []

def combine_associated_files(folder_path, label):
    for file in folder_path.iterdir():
        if file.is_file():
            try:
                text = file.read_text(errors="ignore")
                rows.append({
                    "text": text,
                    "label": label
                })
            except Exception:
                pass
    else:
        for subfolder in folder_path.iterdir():
            if subfolder.is_dir():
                combine_associated_files(subfolder, label)

for folder_path in DOWNLOAD_DIR.iterdir():
    if not folder_path.is_dir():
        continue

    folder_name = folder_path.name.lower()
    
   

    if "ham" in folder_name:
        label = 0
    elif "spam" in folder_name:
        label = 1
    else:
        continue
    
    print(f"Processing folder: {folder_name} is labeled as {label}")

    combine_associated_files(folder_path, label)
            

df = pd.DataFrame(rows)

df.head()


Processing folder: 20030228_easy_ham_2 is labeled as 0
Processing folder: 20030228_hard_ham is labeled as 0
Processing folder: 20030228_spam is labeled as 1
Processing folder: 20030228_easy_ham is labeled as 0
Processing folder: 20030228_spam_2 is labeled as 1
Processing folder: 20021010_spam is labeled as 1
Processing folder: 20021010_hard_ham is labeled as 0
Processing folder: 20021010_easy_ham is labeled as 0
Processing folder: 20050311_spam_2 is labeled as 1


,text,label
0,From rpm-list-admin@freshrpms.net Mon Jul 22 ...,0
1,From webmake-talk-admin@lists.sourceforge.net ...,0
2,From fork-admin@xent.com Wed Aug 14 11:01:15 ...,0
3,From ilug-admin@linux.ie Mon Jul 22 19:50:20 ...,0
4,From razor-users-admin@lists.sourceforge.net ...,0


In [5]:
X = df["text"]
y = df["label"]

print(f"Total samples: {len(df)} Split into 8:2 train-test ratio.")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Total samples: 10751 Split into 8:2 train-test ratio.


In [6]:
vectorizer = TfidfVectorizer(stop_words="english")
X_train_vectorized = vectorizer.fit_transform(X_train)


X_train_vectorized.shape, y_train.shape

((8600, 182225), (8600,))

In [7]:
from sklearn.linear_model import LogisticRegression

log_ref = LogisticRegression()
log_ref.fit(X_train_vectorized, y_train)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [ ]:
from sklearn.metrics import classification_report

X_test_vectorized = vectorizer.transform(X_test)

y_test_pred = log_ref.predict(X_test_vectorized)

print(classification_report(y_test, y_test_pred, target_names=["Ham", "Spam"]))

              precision    recall  f1-score   support

         Ham       0.98      0.99      0.99      1391
        Spam       0.99      0.96      0.97       760

    accuracy                           0.98      2151
   macro avg       0.98      0.98      0.98      2151
weighted avg       0.98      0.98      0.98      2151



In [10]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train_vectorized, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [11]:
y_test_pred = rf_clf.predict(X_test_vectorized)
print(classification_report(y_test, y_test_pred, target_names=["Ham", "Spam"]))

              precision    recall  f1-score   support

         Ham       0.99      1.00      0.99      1391
        Spam       0.99      0.99      0.99       760

    accuracy                           0.99      2151
   macro avg       0.99      0.99      0.99      2151
weighted avg       0.99      0.99      0.99      2151



In [18]:
# Retrieve built-in importances
feature_names = vectorizer.get_feature_names_out()


# Create a clean DataFrame mapped to feature names
feature_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_clf.feature_importances_
}).sort_values(by='Importance', ascending=False)

feature_imp_df.head(10)

,Feature,Importance
59596,beenthere,0.011062
156232,single,0.009618
123971,localhost,0.008213
164785,tr,0.007915
108082,href,0.007284
11834,127,0.006129
150034,request,0.006033
142351,plain,0.005816
179532,yyyy,0.005731
119425,keywords,0.005713


In [ ]:
y_test_pred = rf_clf.predict(X_test_vectorized)
print(classification_report(y_test, y_test_pred, target_names=["Ham", "Spam"]))